# 14 — Ax Bayesian optimization for T-163 joint arrival calibration

**Ticket:** T-163 corridor-mixture joint calibration.

Tune **(p_short, q10, delta_c)** on the embedded Abdella arrival model using [Ax](https://ax.dev/) instead of exhaustive grid search.

Each Ax trial:
- applies `apply_config` (mixture weight, q10, leg setpoint shift, η_ref rebuild via `refresh_filter_laws`)
- **hard-rejects** candidates with analytical **ac2_19** margin ≤ 0
- reports **session_f**, truth-band **p50**, and **pct_60_90** (mean ± SEM over K seeds)
- optionally runs expensive **ac2_11a_ratio** on promising trials only

**Defaults are smoke-sized** (`K=2`, 10 trials). Set `FULL_RUN = True` for 60 trials / K=6.

> **Grid examples are legacy/diagnostic only** — do not run `t163_joint_fast_grid` or
> `t163_joint_constraint_search` for calibration; use this notebook or
> `scripts/run_arrival_calib_bo.py`.

## Setup

From the repo root:

```bash
uv sync --extra notebooks
uv run maturin develop --release --manifest-path crates/voi_py/Cargo.toml
uv run jupyter lab
```

CLI smoke (no notebook UI):

```bash
uv run python scripts/run_arrival_calib_bo.py
```

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from ax.api.client import Client
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
from blueberries_voi.experiments.arrival_joint_calib import (
    REJECTED_OBJECTIVE,
    ax_parameter_configs,
    evaluate_joint_calib_trial,
    evaluate_with_replicates,
)

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

FULL_RUN = False
P_SHORT_BOUNDS = (0.5, 0.9)
Q10_BOUNDS = (1.5, 3.0)
DELTA_C_BOUNDS = (-3.0, 1.0)

if FULL_RUN:
    K_BO_SEEDS = 6
    TOTAL_AX_TRIALS = 60
    AX_PARALLELISM = 4
    INCLUDE_AC2_11A = True
else:
    K_BO_SEEDS = 2
    TOTAL_AX_TRIALS = 10
    AX_PARALLELISM = 2
    INCLUDE_AC2_11A = False

EXTRA_AX_TRIALS = 0
RELOAD_AX = False
AX_JSON = REPO_ROOT / "outputs" / "arrival_joint_calib_bo_ax_client.json"
OUTPUT_JSON = REPO_ROOT / "outputs" / "arrival_joint_calib_bo.json"

RNG = np.random.default_rng(20260828)
BO_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_BO_SEEDS)]

rust_fn = getattr(rust_core, "evaluate_joint_calib_trial_py", None) if rust_core else None
print(f"Rust kernel: {rust_available() and rust_fn is not None}")
print(f"search: p_short={P_SHORT_BOUNDS}, q10={Q10_BOUNDS}, delta_c={DELTA_C_BOUNDS}")
print(f"Ax trials={TOTAL_AX_TRIALS}, K={K_BO_SEEDS}, full_run={FULL_RUN}")
print(f"BO seeds: {BO_SEEDS}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

## Per-trial evaluator (Rust `_core`)

Mirrors `crates/voi_core/src/joint_arrival_calib.rs`:
`configured_model` / `apply_config`, analytical `ac2_19_min_margin`,
`truth_band`, and `EngineSession::with_arrival_model` session_f.

In [ ]:
def _completed_trial_count(client: Client) -> int:
    return sum(1 for t in client._experiment.trials.values() if t.status.is_completed)


demo = evaluate_joint_calib_trial(0.7, 2.5, 0.0, BO_SEEDS[0])
print(
    f"smoke p_short=0.7 q10=2.5 delta_c=0: "
    f"ac2_19={demo.ac2_19_margin:.4f} session_f={demo.session_f:.3f} "
    f"p50={demo.p50:.3f} pct_60_90={demo.pct_60_90:.2%} rejected={demo.rejected_ac2_19}"
)

demo_rep = evaluate_with_replicates(0.7, 2.5, 0.0, BO_SEEDS)
print("replicate session_f:", demo_rep["session_f"])

## Ax BO loop

Objective: maximize replicate-mean **session_f** subject to analytical ac2_19 hard reject.
Auxiliary metrics (p50, pct_60_90, ac2_19_margin) are logged for post-hoc filtering.

In [ ]:
if RELOAD_AX and AX_JSON.is_file():
    client = Client.load_from_json_file(str(AX_JSON))
    completed = _completed_trial_count(client)
    trials_to_run = (
        EXTRA_AX_TRIALS if EXTRA_AX_TRIALS > 0 else max(0, TOTAL_AX_TRIALS - completed)
    )
    trial_log: list[dict[str, Any]] = []
    print(f"Reloaded Ax ({completed} done); running {trials_to_run} more")
else:
    client = Client()
    client.configure_experiment(
        name="t163-joint-arrival-calib",
        parameters=ax_parameter_configs(),
    )
    client.configure_optimization(objective="session_f")
    trial_log = []
    trials_to_run = TOTAL_AX_TRIALS

completed = 0
pbar = tqdm(total=trials_to_run, desc="Ax joint arrival calib")
while completed < trials_to_run:
    batch_n = min(AX_PARALLELISM, trials_to_run - completed)
    trials = client.get_next_trials(max_trials=batch_n)
    for trial_index, parameters in trials.items():
        p_short = float(parameters["p_short"])
        q10 = float(parameters["q10"])
        delta_c = float(parameters["delta_c"])
        metrics = evaluate_with_replicates(
            p_short,
            q10,
            delta_c,
            BO_SEEDS,
            include_ac2_11a=INCLUDE_AC2_11A,
        )
        client.complete_trial(
            trial_index=trial_index,
            raw_data={
                "session_f": metrics["session_f"],
                "p50": metrics["p50"],
                "pct_60_90": metrics["pct_60_90"],
                "ac2_19_margin": metrics["ac2_19_margin"],
            },
        )
        rejected = metrics["session_f"][0] <= REJECTED_OBJECTIVE / 2
        trial_log.append(
            {
                "trial_index": int(trial_index),
                "p_short": p_short,
                "q10": q10,
                "delta_c": delta_c,
                "rejected_ac2_19": rejected,
                "mean_session_f": metrics["session_f"][0],
                "sem_session_f": metrics["session_f"][1],
                "mean_p50": metrics["p50"][0],
                "mean_pct_60_90": metrics["pct_60_90"][0],
                "ac2_19_margin": metrics["ac2_19_margin"][0],
            }
        )
    AX_JSON.parent.mkdir(parents=True, exist_ok=True)
    client.save_to_json_file(str(AX_JSON))
    completed += len(trials)
    pbar.update(len(trials))
pbar.close()

best_params, _pred, best_index, _name = client.get_best_parameterization()
print(f"Best trial {best_index}: {dict(best_params)}")

## Diagnostics

In [ ]:
feasible = [t for t in trial_log if not t["rejected_ac2_19"]]
print(f"{len(feasible)}/{len(trial_log)} trials pass ac2_19")

if feasible:
    fig, ax = plt.subplots(figsize=(8, 4))
    xs = [t["mean_session_f"] for t in feasible]
    ys = [t["mean_p50"] for t in feasible]
    cs = [t["ac2_19_margin"] for t in feasible]
    sc = ax.scatter(xs, ys, c=cs, cmap="viridis", s=60)
    ax.set_xlabel("session_f (mean)")
    ax.set_ylabel("truth-band p50 (mean)")
    ax.set_title("Feasible Ax trials")
    fig.colorbar(sc, ax=ax, label="ac2_19 margin")
    plt.show()

## Save results

In [ ]:
payload: dict[str, Any] = {
    "ticket": "T-163",
    "method": "ax_bo",
    "full_run": FULL_RUN,
    "bo_seeds": BO_SEEDS,
    "ax_client_path": str(AX_JSON.relative_to(REPO_ROOT)),
    "best_trial_index": int(best_index),
    "best_params": {
        "p_short": float(best_params["p_short"]),
        "q10": float(best_params["q10"]),
        "delta_c": float(best_params["delta_c"]),
    },
    "legacy_grid_examples": [
        "crates/voi_core/examples/t163_joint_fast_grid.rs",
        "crates/voi_core/examples/t163_joint_constraint_search.rs",
    ],
    "trials": trial_log,
}
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_JSON}")

## Takeaways

1. **Ax BO replaces exhaustive grid search** for joint (p_short, q10, delta_c) calibration.
2. **ac2_19** is an analytical hard reject; stochastic band/session metrics use K seeds.
3. **Grid Rust examples** remain for spot checks only — not the calibration workflow.
4. Do **not** change `arrival_model.json` from this notebook; winners need human review before commit.